In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Hello world

<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato usando i seguenti requisiti.
Consigliamo di usare queste versioni o versioni più recenti.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Questo esempio è composto da due parti. Prima creerai un semplice programma quantistico e lo eseguirai su un'unità di elaborazione quantistica (QPU). Poiché la ricerca quantistica reale richiede programmi molto più robusti, nella seconda sezione ([Scala a grandi numeri di qubit](#scale-to-large-numbers-of-qubits)) porterai il programma semplice a livello di utilità.

## Installa e autenticati
1. Se non hai ancora installato Qiskit, trovi le istruzioni nella guida [Quickstart](/guides/quick-start).

    - Installa Qiskit Runtime per eseguire job su hardware quantistico:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - Configura un ambiente per eseguire Jupyter notebook in locale:

        ```bash
        pip install jupyter
        ```

2. Configura la tua autenticazione per accedere all'hardware quantistico tramite il gratuito [Open Plan](/guides/plans-overview#open-plan).

    (Se hai ricevuto un'email di invito a un account, segui invece i [passaggi per gli utenti invitati](/guides/cloud-setup-invited).)

    - Vai su [IBM Quantum Platform](https://quantum.cloud.ibm.com/) per accedere o creare un account.
         > **Note:** Se ti connetti tramite un server proxy, devi usare Qiskit Runtime v0.44.0 o versioni successive.
    - Genera la tua chiave API (chiamata anche *API token*) nella [dashboard](https://quantum.cloud.ibm.com/), poi copiala in un luogo sicuro.
    - Vai alla pagina [Instances](https://quantum.cloud.ibm.com/instances) e trova l'istanza che vuoi usare. Passa il cursore sul suo CRN e clicca per copiarlo.

    - Salva le tue credenziali in locale con questo codice:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. Ora puoi usare questo codice Python ogni volta che vuoi autenticarti al Qiskit Runtime Service:
    ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        # Run every time you need the service
        service = QiskitRuntimeService()
    ```
> **Info:** Se stai usando un computer pubblico o un altro ambiente non protetto, segui invece le [istruzioni di autenticazione manuale](/guides/cloud-setup-untrusted) per tenere al sicuro le tue credenziali di autenticazione.
## Crea ed esegui un semplice programma quantistico
I quattro passaggi per scrivere un programma quantistico usando i pattern Qiskit sono:

1.  Mappa il problema in un formato nativo quantistico.

2.  Ottimizza i circuiti e gli operatori.

3.  Esegui usando una funzione primitiva quantistica.

4.  Analizza i risultati.

### Passaggio 1. Mappa il problema in un formato nativo quantistico
In un programma quantistico, i *quantum circuit* sono il formato nativo in cui rappresentare le istruzioni quantistiche, mentre gli *operatori* rappresentano gli osservabili da misurare. Quando crei un circuito, di solito crei un nuovo oggetto [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class), poi aggiungi istruzioni in sequenza.
Il seguente blocco di codice crea un circuito che produce un *Bell state*, ovvero uno stato in cui due qubit sono completamente entangled tra loro.

> **Note:** Il Qiskit SDK usa la numerazione dei bit LSb 0, in cui l'$n^{th}$ cifra ha valore $1 \ll n$ o $2^n$. Per ulteriori dettagli, vedi l'argomento [Bit-ordering in the Qiskit SDK](/guides/bit-ordering).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg)

Consulta [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) nella documentazione per tutte le operazioni disponibili.
Quando crei quantum circuit, devi anche considerare che tipo di dati vuoi ricevere dopo l'esecuzione. Qiskit offre due modi per restituire dati: puoi ottenere una distribuzione di probabilità per un insieme di qubit che scegli di misurare, oppure puoi ottenere il valore di aspettazione di un osservabile. Prepara il tuo workload per misurare il tuo circuito in uno di questi due modi con le [primitive Qiskit](/guides/get-started-with-primitives) (spiegate in dettaglio nel [Passaggio 3](#step-3-execute-using-the-quantum-primitives)).

Questo esempio misura i valori di aspettazione usando il sottomodulo `qiskit.quantum_info`, specificato tramite operatori (oggetti matematici usati per rappresentare un'azione o un processo che modifica uno stato quantistico). Il seguente blocco di codice crea sei operatori di Pauli a due qubit: `IZ`, `IX`, `ZI`, `XI`, `ZZ` e `XX`.

In [2]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

> **Note:** Qui, un operatore come `ZZ` è una notazione abbreviata per il prodotto tensoriale $Z\otimes Z$, che significa misurare Z sul qubit 1 e Z sul qubit 0 insieme, ottenendo informazioni sulla correlazione tra qubit 1 e qubit 0. Valori di aspettazione come questo si scrivono tipicamente anche come $\langle Z_1 Z_0 \rangle$.
> 
> Se lo stato è entangled, allora la misurazione di $\langle Z_1 Z_0 \rangle$ dovrebbe essere diversa dalla misurazione di $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. Per lo specifico stato entangled creato dal nostro circuito descritto sopra, la misurazione di $\langle Z_1 Z_0 \rangle$ dovrebbe essere 1 e la misurazione di $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ dovrebbe essere zero.
<span id="optimize"></span>

### Passaggio 2. Ottimizza i circuiti e gli operatori

Quando si eseguono circuito su un dispositivo, è importante ottimizzare l'insieme di istruzioni contenute nel circuito e minimizzare la profondità complessiva (approssimativamente il numero di istruzioni) del circuito. Questo assicura di ottenere i migliori risultati possibili riducendo gli effetti di errori e rumore. Inoltre, le istruzioni del circuito devono essere conformi all'[Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) del dispositivo backend e devono tenere conto dei gate base e della connettività dei qubit del dispositivo.

Il seguente codice istanzia un dispositivo reale a cui inviare un job e trasforma il circuito e gli osservabili per corrispondere all'ISA di quel backend. Richiede che tu abbia già [salvato le tue credenziali](/guides/cloud-setup)

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### Passaggio 3. Esegui usando le primitive quantistiche
I computer quantistici possono produrre risultati casuali, quindi di solito si raccoglie un campione degli output eseguendo il circuito molte volte. Puoi stimare il valore dell'osservabile usando la classe `Estimator`. `Estimator` è una delle due [primitive](/guides/get-started-with-primitives); l'altra è `Sampler`, che può essere usata per ottenere dati da un computer quantistico. Questi oggetti possiedono un metodo `run()` che esegue la selezione di circuito, osservabili e parametri (se applicabile), usando un [primitive unified bloc (PUB).](/guides/primitives#sampler)

In [4]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d5k96q4jt3vs73ds5tgg


After a job is submitted, you can wait until either the job is completed within your current python instance, or use the `job_id` to retrieve the data at a later time.  (See the [section on retrieving jobs](/docs/guides/monitor-job#retrieve-job-results-at-a-later-time) for details.)

After the job completes, examine its output through the job's `result()` attribute.

In [5]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [6]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see [Get started with Sampler.](/docs/guides/get-started-with-primitives#get-started-with-sampler)

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

> **Note:** Quando esegui il tuo programma quantistico su un dispositivo reale, il tuo workload deve attendere in una coda prima di essere eseguito. Per risparmiare tempo, puoi invece usare il seguente codice per eseguire questo piccolo workload sul [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) con la modalità di test locale di Qiskit Runtime. Nota che questo è possibile solo per un circuito piccolo. Quando aumenti la scala nella sezione successiva, dovrai usare un dispositivo reale.
> 
>   ```python
> 
>   # Use the following code instead if you want to run on a simulator:
> 
>   from qiskit_ibm_runtime.fake_provider import FakeBelemV2
>   backend = FakeBelemV2()
>   estimator = Estimator(backend)
> 
>   # Convert to an ISA circuit and layout-mapped observables.
> 
>   pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
>   isa_circuit = pm.run(qc)
>   mapped_observables = [
>       observable.apply_layout(isa_circuit.layout) for observable in observables
>   ]
> 
>   job = estimator.run([(isa_circuit, mapped_observables)])
>   result = job.result()
> 
>   # This is the result of the entire submission.  You submitted one Pub,
>   # so this contains one inner result (and some metadata of its own).
> 
>   job_result = job.result()
> 
>   # This is the result from our single pub, which had five observables,
>   # so contains information on all five.
> 
>   pub_result = job.result()[0]
>   ```
### Passaggio 4. Analizza i risultati
Il passaggio di analisi è tipicamente quello in cui potresti post-elaborare i tuoi risultati usando, ad esempio, la mitigazione degli errori di misurazione o l'estrapolazione a rumore zero (ZNE). Potresti inserire questi risultati in un altro workflow per ulteriori analisi o preparare un grafico dei valori e dei dati chiave. In generale, questo passaggio è specifico per il tuo problema. Per questo esempio, traccia ciascuno dei valori di aspettazione misurati per il nostro circuito.

I valori di aspettazione e le deviazioni standard per gli osservabili specificati all'Estimator sono accessibili tramite gli attributi `PubResult.data.evs` e `PubResult.data.stds` del risultato del job. Per ottenere i risultati da Sampler, usa la funzione `PubResult.data.meas.get_counts()`, che restituirà un `dict` di misurazioni nella forma di bitstring come chiavi e conteggi come valori corrispondenti. Per ulteriori informazioni, vedi [Inizia con Sampler.](/guides/get-started-with-primitives#get-started-with-sampler)

In [8]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

Nota che per i qubit 0 e 1, i valori di aspettazione indipendenti sia di X che di Z sono 0, mentre le correlazioni (`XX` e `ZZ`) sono 1. Questo è un segno distintivo dell'entanglement quantistico.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with two qubits (first argument) and two classical
# bits (second argument)
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

## Scala a grandi numeri di qubit
Nel calcolo quantistico, il lavoro a scala di utilità è fondamentale per fare progressi nel campo. Tale lavoro richiede computazioni su scala molto più grande; si lavora con circuito che potrebbero usare oltre 100 qubit e oltre 1000 gate. Questo esempio dimostra come puoi svolgere lavoro a scala di utilità su QPU IBM&reg; creando e analizzando uno stato GHZ a 100 qubit. Usa il workflow dei pattern Qiskit e termina misurando il valore di aspettazione $\langle Z_0 Z_i \rangle$ per ciascun qubit.

### Passaggio 1. Mappa il problema
Scrivi una funzione che restituisce un `QuantumCircuit` che prepara uno stato GHZ a $n$ qubit (essenzialmente un Bell state esteso), poi usa quella funzione per preparare uno stato GHZ a 100 qubit e raccogli gli osservabili da misurare.

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

Poi, mappa gli operatori di interesse. Questo esempio usa gli operatori `ZZ` tra i qubit per esaminare il comportamento man mano che si allontanano. Valori di aspettazione sempre più imprecisi (corrotti) tra qubit distanti rivelerebbero il livello di rumore presente.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Step 3. Execute on hardware

Submit the job and enable error suppression by using a technique to reduce errors called [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) The resilience level specifies how much resilience to build against errors. Higher levels generate more accurate results, at the expense of longer processing times.  For further explanation of the options set in the following code, see [Configure error mitigation for Qiskit Runtime.](/docs/guides/configure-error-mitigation)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [13]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d5k9mmqvcahs73a1ni3g


### Passaggio 3. Esegui sull'hardware
Invia il job e abilita la soppressione degli errori usando una tecnica per ridurre gli errori chiamata [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) Il livello di resilienza specifica quanta resilienza costruire contro gli errori. Livelli più alti generano risultati più accurati, a scapito di tempi di elaborazione più lunghi. Per ulteriori spiegazioni sulle opzioni impostate nel seguente codice, vedi [Configura la mitigazione degli errori per Qiskit Runtime.](/guides/configure-error-mitigation)

In [ ]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment.](/docs/guides/online-lab-environments)
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>